<div style="display: flex; background-color: RGB(255,114,0);" >
<h1 style="margin: auto; padding: 30px; ">Produisez une étude de marché avec Python 1/2</h1>
</div>

<b><u>NOTEBOOK 1</u></b>

1.	Pars des données de la FAO (Food and Agriculture Organization) en pièce jointe pour commencer ton analyse puis : 
- Utilise l’analyse PESTEL pour trouver des idées de nouvelles données à ajouter (nous voulons au minimum 8 variables) ;
- Récupère et utilise toutes les données en open data que tu souhaites sur le site de la FAO, de la banque mondiale ou encore sur données mondiales.
2.	Prépare et nettoie les données : 
- Si tu utilises plusieurs sources de données, regroupe-les dans un même fichier. 
- L’idéal serait d’avoir au minimum 100 pays dans notre analyse (qui couvrent au moins 60% de la population mondiale). 
3.	Passe ensuite à l’exploration des données (en Python ou en R).

<b><u>VARIABLES SÉLECTIONNÉES PAR AXE PESTEL</u></b>
1. PIB par habitant (Economique - worldbank)
2. Croissance du PIB par pays (Economique - worldbank)
3. Consommation de poulet par habitant (Socioculturel - world population review) > pas possible car ne s'aligne pas avec les années des autres bdd
4. Import de poulet (Economique - FAO)
5. Indice de stabilité politique (Politique - worldbank)
6. Qualité de l'environnement commercial (Legal - worldbank)
7. Distance entre la france et les autres pays (Environnemental - cepii)
8. Part de la population urbaine (Socioculturel - worldbank)



<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des librairies et chargement des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.1 - Importation des librairies</h3>
</div>

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.colors as mcolors
from scipy.stats import pearsonr

In [6]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.2 - Chargements des fichiers</h3>
</div>

In [8]:
df_pop = pd.read_excel("p11-pop.xlsx")
df_pib = pd.read_excel("p11-pib.xlsx")
df_croissance = pd.read_excel("p11-croissance.xlsx")
df_export = pd.read_excel("p11-exports.xlsx")
df_import = pd.read_excel("p11-imports.xlsx")
df_production = pd.read_excel("p11-production.xlsx")
df_stabilite = pd.read_excel("p11-stabilite.xlsx")
df_popurb = pd.read_excel("p11-urbanisation.xlsx")
df_distance = pd.read_excel("p11-distance.xlsx")
df_business = pd.read_excel("p11-business.xlsx")

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 2 - Analyse exploratoire, pré-traitement et jointure des datasets</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.0 - Années disponibles par dataset</h3>
</div>

In [11]:
#POPULATION (192 pays reconnus par l'onu)
df_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Country Name  192 non-null    object
 1   Country Code  192 non-null    object
 2   2016          192 non-null    int64 
 3   2017          192 non-null    int64 
 4   2018          192 non-null    int64 
 5   2019          192 non-null    int64 
 6   2020          192 non-null    int64 
 7   2021          192 non-null    int64 
 8   2022          192 non-null    int64 
 9   2023          192 non-null    int64 
 10  2024          192 non-null    int64 
dtypes: int64(9), object(2)
memory usage: 16.6+ KB


In [12]:
#PIB (2018 et 2019 semblent être les années les plus récentes contenant le plus de valeurs hors années covid)
df_pib.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    object 
 1   Country Code    266 non-null    object 
 2   Indicator Name  266 non-null    object 
 3   Indicator Code  266 non-null    object 
 4   2015            259 non-null    float64
 5   2016            258 non-null    float64
 6   2017            258 non-null    float64
 7   2018            258 non-null    float64
 8   2019            258 non-null    float64
 9   2020            257 non-null    float64
 10  2021            257 non-null    float64
 11  2022            256 non-null    float64
 12  2023            248 non-null    float64
 13  2024            231 non-null    float64
dtypes: float64(10), object(4)
memory usage: 29.2+ KB


In [13]:
#CROISSANCE (2018 semble être l'année contenant le plus de valeurs)
df_croissance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    object 
 1   Country Code    266 non-null    object 
 2   Indicator Name  266 non-null    object 
 3   Indicator Code  266 non-null    object 
 4   2015            258 non-null    float64
 5   2016            257 non-null    float64
 6   2017            257 non-null    float64
 7   2018            258 non-null    float64
 8   2019            257 non-null    float64
 9   2020            257 non-null    float64
 10  2021            257 non-null    float64
 11  2022            256 non-null    float64
 12  2023            249 non-null    float64
 13  2024            232 non-null    float64
dtypes: float64(10), object(4)
memory usage: 29.2+ KB


In [14]:
#1. EXPORT
df_export.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1047 entries, 0 to 1046
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       1047 non-null   object 
 1   Domain            1047 non-null   object 
 2   Area Code (M49)   1047 non-null   int64  
 3   Country code      1047 non-null   object 
 4   Area              1047 non-null   object 
 5   Element Code      1047 non-null   int64  
 6   Element           1047 non-null   object 
 7   Item Code (CPC)   1047 non-null   int64  
 8   Item              1047 non-null   object 
 9   Year Code         1047 non-null   int64  
 10  Year              1047 non-null   int64  
 11  Unit              1047 non-null   object 
 12  Value             1047 non-null   float64
 13  Flag              1047 non-null   object 
 14  Flag Description  1047 non-null   object 
 15  Note              110 non-null    object 
dtypes: float64(1), int64(5), object(10)
memory

In [15]:
#2. EXPORT année
df_export['Year'].unique()

array([2017, 2018, 2019, 2020, 2021, 2022, 2023, 2016], dtype=int64)

In [16]:
#3. EXPORT année 2018
df_export_2018 = df_export[df_export['Year'] == 2018]
df_export_2018.info()

<class 'pandas.core.frame.DataFrame'>
Index: 133 entries, 1 to 1041
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       133 non-null    object 
 1   Domain            133 non-null    object 
 2   Area Code (M49)   133 non-null    int64  
 3   Country code      133 non-null    object 
 4   Area              133 non-null    object 
 5   Element Code      133 non-null    int64  
 6   Element           133 non-null    object 
 7   Item Code (CPC)   133 non-null    int64  
 8   Item              133 non-null    object 
 9   Year Code         133 non-null    int64  
 10  Year              133 non-null    int64  
 11  Unit              133 non-null    object 
 12  Value             133 non-null    float64
 13  Flag              133 non-null    object 
 14  Flag Description  133 non-null    object 
 15  Note              12 non-null     object 
dtypes: float64(1), int64(5), object(10)
memory usage

In [17]:
#1. IMPORT
df_import.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       1530 non-null   object 
 1   Domain            1530 non-null   object 
 2   Area Code (M49)   1530 non-null   int64  
 3   Country code      1530 non-null   object 
 4   Area              1530 non-null   object 
 5   Element Code      1530 non-null   int64  
 6   Element           1530 non-null   object 
 7   Item Code (CPC)   1530 non-null   int64  
 8   Item              1530 non-null   object 
 9   Year Code         1530 non-null   int64  
 10  Year              1530 non-null   int64  
 11  Unit              1530 non-null   object 
 12  Value             1530 non-null   float64
 13  Flag              1530 non-null   object 
 14  Flag Description  1530 non-null   object 
 15  Note              342 non-null    object 
dtypes: float64(1), int64(5), object(10)
memory

In [18]:
#2. IMPORT année
df_import['Year'].unique()

array([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023], dtype=int64)

In [19]:
#3. IMPORT année 2018
df_import_2018 = df_import[df_import['Year'] == 2018]
df_import_2018.info()

<class 'pandas.core.frame.DataFrame'>
Index: 191 entries, 2 to 1524
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       191 non-null    object 
 1   Domain            191 non-null    object 
 2   Area Code (M49)   191 non-null    int64  
 3   Country code      191 non-null    object 
 4   Area              191 non-null    object 
 5   Element Code      191 non-null    int64  
 6   Element           191 non-null    object 
 7   Item Code (CPC)   191 non-null    int64  
 8   Item              191 non-null    object 
 9   Year Code         191 non-null    int64  
 10  Year              191 non-null    int64  
 11  Unit              191 non-null    object 
 12  Value             191 non-null    float64
 13  Flag              191 non-null    object 
 14  Flag Description  191 non-null    object 
 15  Note              42 non-null     object 
dtypes: float64(1), int64(5), object(10)
memory usage

In [20]:
#1. PRODUCTION
df_production.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1171 entries, 0 to 1170
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Code Domaine            1171 non-null   object 
 1   Domaine                 1171 non-null   object 
 2   Code zone (M49)         1171 non-null   int64  
 3   Country code            1171 non-null   object 
 4   Zone                    1171 non-null   object 
 5   Code Élément            1171 non-null   int64  
 6   Élément                 1171 non-null   object 
 7   Code Produit (CPC)      1171 non-null   int64  
 8   Produit                 1171 non-null   object 
 9   Code année              1171 non-null   int64  
 10  Année                   1171 non-null   int64  
 11  Unité                   1171 non-null   object 
 12  Valeur                  1171 non-null   float64
 13  Symbole                 1171 non-null   object 
 14  Description du Symbole  1171 non-null   

In [21]:
#2. PRODUCTION année
df_production['Année'].unique()

array([2018, 2019, 2020, 2021, 2022, 2023], dtype=int64)

In [22]:
#3. PRODUCTION année 2018
df_production_2018 = df_production[df_production['Année'] == 2018]
df_production_2018.info()

<class 'pandas.core.frame.DataFrame'>
Index: 196 entries, 0 to 1165
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Code Domaine            196 non-null    object 
 1   Domaine                 196 non-null    object 
 2   Code zone (M49)         196 non-null    int64  
 3   Country code            196 non-null    object 
 4   Zone                    196 non-null    object 
 5   Code Élément            196 non-null    int64  
 6   Élément                 196 non-null    object 
 7   Code Produit (CPC)      196 non-null    int64  
 8   Produit                 196 non-null    object 
 9   Code année              196 non-null    int64  
 10  Année                   196 non-null    int64  
 11  Unité                   196 non-null    object 
 12  Valeur                  196 non-null    float64
 13  Symbole                 196 non-null    object 
 14  Description du Symbole  196 non-null    object

In [23]:
#STABILITE
df_stabilite.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    object 
 1   Country Code    266 non-null    object 
 2   Indicator Name  266 non-null    object 
 3   Indicator Code  266 non-null    object 
 4   2015            205 non-null    float64
 5   2016            205 non-null    float64
 6   2017            205 non-null    float64
 7   2018            205 non-null    float64
 8   2019            205 non-null    float64
 9   2020            205 non-null    float64
 10  2021            205 non-null    float64
 11  2022            205 non-null    float64
 12  2023            205 non-null    float64
 13  2024            0 non-null      float64
dtypes: float64(10), object(4)
memory usage: 29.2+ KB


In [24]:
#URBANISATION
df_popurb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    object 
 1   Country Code    266 non-null    object 
 2   Indicator Name  266 non-null    object 
 3   Indicator Code  266 non-null    object 
 4   2015            263 non-null    float64
 5   2016            263 non-null    float64
 6   2017            263 non-null    float64
 7   2018            263 non-null    float64
 8   2019            263 non-null    float64
 9   2020            263 non-null    float64
 10  2021            263 non-null    float64
 11  2022            263 non-null    float64
 12  2023            263 non-null    float64
 13  2024            263 non-null    float64
dtypes: float64(10), object(4)
memory usage: 29.2+ KB


In [25]:
#BUSINESS
df_business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Country Name   191 non-null    object 
 1   Country Code   191 non-null    object 
 2   Series Name    191 non-null    object 
 3   Series Code    191 non-null    object 
 4   2016 [YR2016]  191 non-null    object 
 5   2017 [YR2017]  191 non-null    object 
 6   2018 [YR2018]  191 non-null    float64
 7   2019 [YR2019]  191 non-null    float64
dtypes: float64(2), object(6)
memory usage: 12.1+ KB


In [26]:
#DISTANCE
df_distance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   iso_o   224 non-null    object 
 1   iso_d   224 non-null    object 
 2   dist    224 non-null    float64
dtypes: float64(1), object(2)
memory usage: 5.4+ KB


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1 - Nettoyage</h3>
</div>

In [28]:
#POPULATION 2018 (colonnes principales - clé : Country code)
df_pop_2018 = df_pop[['Country Name','Country Code', '2018']]
df_pop_2018.head()

,Country Name,Country Code,2018
0,Afghanistan,AFG,36743039
1,Angola,AGO,31297155
2,Albania,ALB,2866376
3,Andorra,AND,75162
4,United Arab Emirates,ARE,9346701


In [29]:
#PIB 2018
df_pib_2018 = df_pib[['Country Code', '2018']]
df_pib_2018.head()

,Country Code,2018
0,ABW,3.276184e+09
1,AFE,1.012291e+12
2,AFG,1.805322e+10
3,AFW,7.778404e+11
4,AGO,7.945069e+10


In [30]:
#CROISSANCE 2018 (% de croissance du pib)
df_croissance_2018 = df_croissance[['Country Code', '2018']]
df_croissance_2018.head()

,Country Code,2018
0,ABW,2.397086
1,AFE,2.665038
2,AFG,1.189228
3,AFW,2.904654
4,AGO,-1.316362


In [31]:
#EXPORTS 2018 (en tonnes)
df_exp_2018 = df_export_2018[['Country code', 'Value']]
df_exp_2018.head()

,Country code,Value
1,AFG,154.00
7,DZA,406.64
15,AGO,34.90
22,ATG,6.54
30,ARG,162443.43


In [32]:
#IMPORTS 2018 (en tonnes)
df_imp_2018 = df_import_2018[['Country code', 'Value']]
df_imp_2018.head()

,Country code,Value
2,AFG,23913.00
10,ALB,11587.51
18,DZA,26.74
25,AGO,322679.69
33,ATG,5943.95


In [33]:
#PRODUCTION 2018 (en tonnes)
df_prod_2018 = df_production_2018[['Country code', 'Valeur']]
df_prod_2018.head()

,Country code,Valeur
0,AFG,29315.76
6,ZAF,1743285.00
12,ALB,15587.27
18,DZA,377379.70
24,DEU,1021000.00


In [34]:
#STABILITE 2018 (indice -2+2)
df_stabilite_2018 = df_stabilite[['Country Code', '2018']]
df_stabilite_2018.head()

,Country Code,2018
0,ABW,1.337510
1,AFE,NaN
2,AFG,-2.753441
3,AFW,NaN
4,AGO,-0.347209


In [35]:
#URBANISATION 2018 (% de la pop vivant en milieu urbain)
df_popurb_2018 = df_popurb[['Country Code', '2018']]
df_popurb_2018.head()

,Country Code,2018
0,ABW,43.411000
1,AFE,35.893398
2,AFG,25.495000
3,AFW,46.709753
4,AGO,65.514000


In [36]:
#BUSINESS 2018 (échelle 0-100)
df_business_2018 = df_business[['Country Code', '2018 [YR2018]']]
df_business_2018.head()

,Country Code,2018 [YR2018]
0,NZL,87.00452
1,SGP,85.84348
2,DNK,85.17028
3,HKG,85.04858
4,USA,83.57395


In [37]:
#DISTANCE (km)
df_dist = df_distance[['iso_d', 'dist']]
df_dist.head()

,iso_d,dist
0,ABW,7685.884
1,AFG,5590.381
2,AGO,6510.322
3,AIA,6710.570
4,ALB,1603.534


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.2 - Jointure</h3>
</div>

In [39]:
# Normalisation des noms des clés primaires
df_exp_2018 = df_exp_2018.rename(columns={'Country code': 'Country Code'})
df_imp_2018 = df_imp_2018.rename(columns={'Country code': 'Country Code'})
df_prod_2018 = df_prod_2018.rename(columns={'Country code': 'Country Code'})
df_dist = df_dist.rename(columns={'iso_d': 'Country Code'})

In [40]:
# Renommer colonnes qui portent le même nom
df_pop_2018 = df_pop_2018.rename(columns={'2018': 'population'})
df_pib_2018 = df_pib_2018.rename(columns={'2018': 'pib'})
df_croissance_2018 = df_croissance_2018.rename(columns={'2018': 'croissance'})
df_exp_2018 = df_exp_2018.rename(columns={'Value': 'export'})
df_imp_2018 = df_imp_2018.rename(columns={'Value': 'import'})
df_prod_2018 = df_prod_2018.rename(columns={'Valeur': 'production'})
df_stabilite_2018 = df_stabilite_2018.rename(columns={'2018': 'stabilité'})
df_popurb_2018 = df_popurb_2018.rename(columns={'2018': 'urbanisation'})
df_business_2018 = df_business_2018.rename(columns={'2018 [YR2018]': 'doing business'})
df_dist = df_dist.rename(columns={'dist': 'distance'})

In [41]:
#jointures left à partir de df_pop_2018 pour garder toutes ses lignes
df_final = df_pop_2018.copy()

df_final = df_final.merge(df_pib_2018, on='Country Code', how='left')
df_final = df_final.merge(df_croissance_2018, on='Country Code', how='left')
df_final = df_final.merge(df_exp_2018, on='Country Code', how='left')
df_final = df_final.merge(df_imp_2018, on='Country Code', how='left')
df_final = df_final.merge(df_prod_2018, on='Country Code', how='left')
df_final = df_final.merge(df_stabilite_2018, on='Country Code', how='left')
df_final = df_final.merge(df_popurb_2018, on='Country Code', how='left')
df_final = df_final.merge(df_business_2018, on='Country Code', how='left')
df_final = df_final.merge(df_dist, on='Country Code', how='left')

In [42]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    193 non-null    object 
 1   Country Code    193 non-null    object 
 2   population      193 non-null    int64  
 3   pib             189 non-null    float64
 4   croissance      188 non-null    float64
 5   export          128 non-null    float64
 6   import          181 non-null    float64
 7   production      185 non-null    float64
 8   stabilité       193 non-null    float64
 9   urbanisation    193 non-null    float64
 10  doing business  186 non-null    float64
 11  distance        185 non-null    float64
dtypes: float64(9), int64(1), object(2)
memory usage: 18.2+ KB


In [43]:
#liste pays sans correspondance df_pop_2018 (toutes colonnes confondues) OU ayant des valeurs manquantes
colonnes_a_verifier = ['pib', 'croissance', 'export', 'import', 'production', 'stabilité', 'urbanisation', 'doing business', 'distance']

# Filtrer les lignes où au moins une des colonnes est manquante
df_manquants = df_final[df_final[colonnes_a_verifier].isnull().any(axis=1)]

# Afficher les Country Code concernés
print(df_manquants['Country Code'].tolist())

['ALB', 'AND', 'BDI', 'BFA', 'BHS', 'BTN', 'CAF', 'CMR', 'COG', 'COM', 'CPV', 'CUB', 'DJI', 'DMA', 'ECU', 'ERI', 'ETH', 'FSM', 'GAB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRD', 'GUY', 'HTI', 'ISR', 'KIR', 'KNA', 'LBY', 'LCA', 'LIE', 'LSO', 'MCO', 'MDG', 'MDV', 'MHL', 'MLI', 'MNE', 'MNG', 'MOZ', 'MRT', 'MUS', 'NER', 'NPL', 'NRU', 'NZL', 'PLW', 'PRK', 'PRY', 'PSE', 'QAT', 'ROU', 'SLB', 'SLE', 'SMR', 'SOM', 'SRB', 'SSD', 'STP', 'SYC', 'TCD', 'TJK', 'TKM', 'TLS', 'TON', 'TUV', 'UZB', 'VCT', 'VEN', 'VUT', 'YEM', 'ZWE']


In [44]:
#liste pays sans correspondance df_pop_2018 par colonne OU ayant des valeurs manquantes
colonnes_a_verifier = ['pib', 'croissance', 'export', 'import', 'production', 'stabilité', 'urbanisation', 'doing business', 'distance']

manquants_par_colonne = {}

for col in colonnes_a_verifier:
    pays_manquants = df_final[df_final[col].isnull()]['Country Code'].tolist()
    manquants_par_colonne[col] = pays_manquants

# Affichage
for col, pays in manquants_par_colonne.items():
    print(f"{col} ({len(pays)} pays manquants) : {pays}")

pib (4 pays manquants) : ['ERI', 'PRK', 'SSD', 'VEN']
croissance (5 pays manquants) : ['ERI', 'LIE', 'PRK', 'SSD', 'VEN']
export (65 pays manquants) : ['ALB', 'AND', 'BDI', 'BFA', 'BHS', 'BTN', 'CAF', 'CMR', 'COG', 'COM', 'CPV', 'DJI', 'DMA', 'ECU', 'ERI', 'ETH', 'FSM', 'GAB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRD', 'GUY', 'HTI', 'KIR', 'KNA', 'LBY', 'LCA', 'LIE', 'LSO', 'MCO', 'MDG', 'MDV', 'MHL', 'MLI', 'MNG', 'MOZ', 'MRT', 'MUS', 'NER', 'NPL', 'NRU', 'PLW', 'PRK', 'PSE', 'QAT', 'SLB', 'SLE', 'SMR', 'SOM', 'SSD', 'STP', 'SYC', 'TCD', 'TJK', 'TKM', 'TLS', 'TON', 'TUV', 'UZB', 'VCT', 'VUT', 'YEM', 'ZWE']
import (12 pays manquants) : ['AND', 'ECU', 'FSM', 'ISR', 'LIE', 'MCO', 'MDG', 'MHL', 'NZL', 'PLW', 'PRY', 'SMR']
production (8 pays manquants) : ['AND', 'DJI', 'LIE', 'MCO', 'MDV', 'MHL', 'PLW', 'SMR']
stabilité (0 pays manquants) : []
urbanisation (0 pays manquants) : []
doing business (7 pays manquants) : ['AND', 'CUB', 'MCO', 'NRU', 'PRK', 'TKM', 'TUV']
distance (8 pays manquants) : ['L

In [45]:
df_complet = df_final.dropna()
df_complet.info()

<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 0 to 191
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    120 non-null    object 
 1   Country Code    120 non-null    object 
 2   population      120 non-null    int64  
 3   pib             120 non-null    float64
 4   croissance      120 non-null    float64
 5   export          120 non-null    float64
 6   import          120 non-null    float64
 7   production      120 non-null    float64
 8   stabilité       120 non-null    float64
 9   urbanisation    120 non-null    float64
 10  doing business  120 non-null    float64
 11  distance        120 non-null    float64
dtypes: float64(9), int64(1), object(2)
memory usage: 12.2+ KB


In [46]:
df_complet.head()

,Country Name,Country Code,population,pib,croissance,export,import,production,stabilité,urbanisation,doing business,distance
0,Afghanistan,AFG,36743039,1.805322e+10,1.189228,154.00,23913.00,29315.76,-2.753441,25.495,44.20343,5590.381
1,Angola,AGO,31297155,7.945069e+10,-1.316362,34.90,322679.69,41476.19,-0.347209,65.514,41.20205,6510.322
4,United Arab Emirates,ARE,9346701,4.270494e+11,1.313914,36433.62,591186.03,47271.00,0.689256,86.522,81.58883,5249.535
5,Argentina,ARG,44654882,5.248199e+11,-2.617396,162443.43,6623.56,2068494.00,0.006910,91.870,58.18388,11072.250
6,Armenia,ARM,2969000,1.245794e+10,5.200000,73.72,32212.25,12300.00,-0.451067,63.149,73.19187,3434.071


In [47]:
total_population = df_complet['population'].sum()
print(f"Population totale couverte : {total_population:,}")

Population totale couverte : 8,287,344,454


In [88]:
df_chine = df_complet[df_complet['Country Code'] == 'CHN']
print(df_chine)

   Country Name Country Code  population           pib  croissance     export  \
32        China          CHN  1402760000  1.414777e+13    6.756718  177284.51   
33        China          CHN  1402760000  1.414777e+13    6.756718  177284.51   

       import   production  stabilité  urbanisation  doing business  distance  
32  502221.09  14583728.12  -0.298572        59.152        73.30366  8225.232  
33  502221.09  13958000.00  -0.298572        59.152        73.30366  8225.232  


In [90]:
doublons = df_complet[df_complet.duplicated(subset='Country Code', keep=False)]
print(doublons)

   Country Name Country Code  population           pib  croissance     export  \
32        China          CHN  1402760000  1.414777e+13    6.756718  177284.51   
33        China          CHN  1402760000  1.414777e+13    6.756718  177284.51   

       import   production  stabilité  urbanisation  doing business  distance  
32  502221.09  14583728.12  -0.298572        59.152        73.30366  8225.232  
33  502221.09  13958000.00  -0.298572        59.152        73.30366  8225.232  


In [48]:
df_complet.to_csv("df_complet.csv", sep=';', index=False)